## Normalization vs Standardization — Hands-on
This notebook reproduces the exact workflow we discussed: loading your `Housing.csv` (from `../Data/Housing.csv`),
training a linear regression model, and demonstrating the effects of MinMax (normalization) and Standard (z-score) scaling.

**Notes:**
- Make sure the active kernel is the same virtual environment that contains `scikit-learn` and `statsmodels`.
- If packages are missing, install them inside your venv: `pip install scikit-learn statsmodels ipykernel`.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import statsmodels.api as sm

print('Libraries imported')

Libraries imported


In [2]:
# 1) Load dataset (adjust path if your structure differs)

csv_path = '../Data/Housing.csv'
print('Loading:', csv_path)
df = pd.read_csv(csv_path)

df.head()

Loading: ../Data/Housing.csv


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [3]:
# 2) Inspect data

df.info()

df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             20 non-null     int64 
 1   area              20 non-null     int64 
 2   bedrooms          20 non-null     int64 
 3   bathrooms         20 non-null     int64 
 4   stories           20 non-null     int64 
 5   mainroad          20 non-null     object
 6   guestroom         20 non-null     object
 7   basement          20 non-null     object
 8   hotwaterheating   20 non-null     object
 9   airconditioning   20 non-null     object
 10  parking           20 non-null     int64 
 11  prefarea          20 non-null     object
 12  furnishingstatus  20 non-null     object
dtypes: int64(6), object(7)
memory usage: 2.2+ KB


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
count,2.000000e+01,20.000000,20.000000,20.000000,20.000000,20,20,20,20,20,20.00000,20,20
unique,NaN,NaN,NaN,NaN,NaN,1,2,2,1,2,NaN,2,3
top,NaN,NaN,NaN,NaN,NaN,yes,no,no,no,yes,NaN,yes,furnished
freq,NaN,NaN,NaN,NaN,NaN,20,18,11,20,15,NaN,12,9
mean,1.016625e+07,7821.500000,3.400000,1.850000,2.200000,NaN,NaN,NaN,NaN,NaN,1.75000,NaN,NaN
std,1.401438e+06,2801.094288,0.753937,0.812728,0.951453,NaN,NaN,NaN,NaN,NaN,0.71635,NaN,NaN
min,8.600000e+06,4320.000000,2.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,0.00000,NaN,NaN
25%,9.112500e+06,6315.000000,3.000000,1.000000,2.000000,NaN,NaN,NaN,NaN,NaN,1.00000,NaN,NaN
50%,9.700000e+06,7420.000000,3.000000,2.000000,2.000000,NaN,NaN,NaN,NaN,NaN,2.00000,NaN,NaN
75%,1.099000e+07,8520.000000,4.000000,2.000000,2.250000,NaN,NaN,NaN,NaN,NaN,2.00000,NaN,NaN


In [4]:
# 3) Drop categorical columns intentionally (we focus on scaling)

categorical_cols = df.select_dtypes(include='object').columns.to_list()
print('Categorical columns (dropped):', categorical_cols)

df_num = df.drop(columns=categorical_cols)

df_num.head()

Categorical columns (dropped): ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']


,price,area,bedrooms,bathrooms,stories,parking
0,13300000,7420,4,2,3,2
1,12250000,8960,4,4,4,3
2,12250000,9960,3,2,2,2
3,12215000,7500,4,2,2,3
4,11410000,7420,4,1,2,2


In [5]:
# 4) Prepare features and target

X = df_num.drop('price', axis=1)
y = df_num['price']

print('Feature columns:', list(X.columns))
print('Number of rows:', len(df))

Feature columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
Number of rows: 20


In [6]:
# 5) Train/test split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42)
print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)

X_train shape: (14, 5)
X_test shape : (6, 5)


In [7]:
# 6) Baseline Linear Regression (no scaling)

lr = LinearRegression()
lr.fit(X_train, y_train)

print('R^2 (no scaling) on TEST set:', lr.score(X_test, y_test))

R^2 (no scaling) on TEST set: 0.33637954423643357


In [8]:
# 7) Statsmodels OLS (no scaling) - for statistical diagnostics

X_train_sm = sm.add_constant(X_train)
ols_raw = sm.OLS(y_train, X_train_sm).fit()

print(ols_raw.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.321
Model:                            OLS   Adj. R-squared:                 -0.103
Method:                 Least Squares   F-statistic:                    0.7575
Date:                Mon, 26 Jan 2026   Prob (F-statistic):              0.604
Time:                        00:20:56   Log-Likelihood:                -212.64
No. Observations:                  14   AIC:                             437.3
Df Residuals:                       8   BIC:                             441.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       6.678e+06   1.92e+06      3.484      0.0

In [9]:
# 8) MinMax Normalization (fit on train only)

minmax = MinMaxScaler()
X_train_norm = minmax.fit_transform(X_train)
X_test_norm = minmax.transform(X_test)

lr_norm = LinearRegression()
lr_norm.fit(X_train_norm, y_train)

print('R^2 (MinMax) on TEST set:', lr_norm.score(X_test_norm, y_test))

R^2 (MinMax) on TEST set: 0.3363795442364328


In [10]:
# 9) Statsmodels OLS on normalized features

X_train_norm_sm = sm.add_constant(X_train_norm)
ols_norm = sm.OLS(y_train, X_train_norm_sm).fit()

print(ols_norm.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.321
Model:                            OLS   Adj. R-squared:                 -0.103
Method:                 Least Squares   F-statistic:                    0.7575
Date:                Mon, 26 Jan 2026   Prob (F-statistic):              0.604
Time:                        00:21:01   Log-Likelihood:                -212.64
No. Observations:                  14   AIC:                             437.3
Df Residuals:                       8   BIC:                             441.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       7.738e+06   1.24e+06      6.256      0.0

In [11]:
# 10) Standardization (Z-score) - fit on train only

std = StandardScaler()
X_train_std = std.fit_transform(X_train)
X_test_std = std.transform(X_test)

lr_std = LinearRegression()
lr_std.fit(X_train_std, y_train)

print('R^2 (StandardScaler) on TEST set:', lr_std.score(X_test_std, y_test))

R^2 (StandardScaler) on TEST set: 0.33637954423643357


In [12]:
# 11) Statsmodels OLS on standardized features

X_train_std_sm = sm.add_constant(X_train_std)
ols_std = sm.OLS(y_train, X_train_std_sm).fit()

print(ols_std.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.321
Model:                            OLS   Adj. R-squared:                 -0.103
Method:                 Least Squares   F-statistic:                    0.7575
Date:                Mon, 26 Jan 2026   Prob (F-statistic):              0.604
Time:                        00:21:06   Log-Likelihood:                -212.64
No. Observations:                  14   AIC:                             437.3
Df Residuals:                       8   BIC:                             441.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       9.934e+06   3.38e+05     29.424      0.0

## Final takeaways

- **Scaling does not change the predictive power of ordinary linear regression** (R² will remain effectively the same) — it changes numeric representation and coefficient scales.
- **Always fit scalers on training data only** to avoid data leakage.
- Use scaling for algorithms that depend on distances or gradient optimization (KNN, SVM, neural networks, regularized regressions).
